**U24AI040 NLP LAB 3 Q2**

Question:
2. You are given all the nouns from the brown corpus (brown_nouns.txt). You need to
design a finite state transducer (FST) to generate the morph/grammatical features for
every word in the corpus. Your output should look like the following:
foxes = fox+N+PL (can be generalized as root+category+number)
fox = fox+N+SG [where SG->Singular, PL->Plural, N->Noun]
You need to take the following properties:
Name Rule Description Example
E insertion e is added after -s, -z, -x,
-ch, -sh before -s is added

watch/watches, fox/foxes

Y replacement -y changes to -ie before -s try/tries
S addition -s is added at the end bag/bags
You need to ensure that incorrect words are not generated. You return an output “Invalid
Word” in that case. Example: foxs = “Invalid Word”

There are techniques to decrease the number of states, try implementing them. Bonus
marks for such techniques. [Don’t try to code the rules, need to design an FST with
transition table, input and output alphabets] 

In [59]:
from pathlib import Path
import re
import string

path = Path("brown_nouns.txt")

data = []

with path.open("r", encoding="utf-8") as file:
    for word in file:
        word = word.strip().lower()

        if word and re.fullmatch(r"[a-z]+", word):
            data.append(word)

word_dict = set(data)

print("Total nouns:", len(data))

Total nouns: 201654


In [60]:
class FST:

    def __init__(
        self,
        initial_state,
        input_symbols,
        output_symbols,
        transition,
        output_function,
        final_state
    ):

        self.initial_state = initial_state
        self.input_symbols = input_symbols
        self.output_symbols = output_symbols
        self.transition = transition
        self.output_function = output_function
        self.final_state = final_state


    def correct(self, res):

        end = res.find("+")
        base_word = res[:end]

        # Get grammatical information
        feature = res[end:]

        if "+invalid" in res:
            return "Invalid Word"

        if "+SG" in res:

            if base_word in word_dict:
                return res

            return "Invalid Word"

        if base_word.endswith("ies"):

            s_word = base_word[:-3] + "y"

            if s_word in word_dict:
                return s_word + feature

        if base_word.endswith("es"):

            s_word = base_word[:-2]

            if s_word in word_dict:

                if (
                    s_word.endswith("s")
                    or s_word.endswith("z")
                    or s_word.endswith("x")
                    or s_word.endswith("ch")
                    or s_word.endswith("sh")
                ):
                    return s_word + feature


        if base_word.endswith("s"):

            s_word = base_word[:-1]

            if s_word in word_dict:

                if(s_word.endswith("s") or s_word.endswith("z") or s_word.endswith("x") or s_word.endswith("ch")  or s_word.endswith("sh")):
                    return "Invalid Word"

                return s_word + feature


        return "Invalid Word"

    def check(self, word):

        word = word.lower().strip()

        if not word:
            return "Invalid Word"

        current_state = self.initial_state
        res_output = []
        i = 0
        while current_state != self.final_state:
            if i >= len(word):
                current_input = ""
            else:
                current_input = word[i]

            i += 1
            if (current_state not in self.transition or current_input not in self.transition[current_state]):
                return "Invalid Word"

            next_state = self.transition[current_state][current_input]

            current_output = self.output_function[current_state][current_input]

            current_state = next_state

            if current_output != "":
                res_output.append(current_output)


        res = "".join(res_output)

        return self.correct(res)

In [61]:
inputs = list(string.ascii_lowercase)

inputs.append("")

outputs = list(string.ascii_lowercase) + [
    "",
    "+N+SG",
    "+N+PL",
    "+invalid"
]

In [ ]:
initial_state = "q1"
final_state = "qend"


states = [
    "q1",#qst
    "q2",#qc
    "q3",#q_special
    "q4",#qe
    "q5",#qes
    "q6",#q_s
    "q7",#q_s_error
    "qend"
]

In [63]:
q1_transitions = {}
q1_output = {}


for ch in inputs:

    if ch not in ["c", "s", "x", "z", ""]:
        q1_transitions[ch] = "q1"
        q1_output[ch] = ch

q1_transitions["c"] = "q2"
q1_output["c"] = "c"

q1_transitions["s"] = "q6"
q1_output["s"] = "s"


for ch in ["x", "z"]:
    q1_transitions[ch] = "q3"
    q1_output[ch] = ch

q1_transitions[""] = "qend"
q1_output[""] = "+N+SG"

In [64]:
q2_transitions = {}
q2_output = {}


for ch in inputs:

    if ch == "h":
        q2_transitions[ch] = "q3"
        q2_output[ch] = "h"

    elif ch == "":
        q2_transitions[ch] = "qend"
        q2_output[ch] = "+N+SG"

    else:
        q2_transitions[ch] = "q1"
        q2_output[ch] = ch

In [65]:
q3_transitions = {}
q3_output = {}


for ch in inputs:

    if ch == "e":
        q3_transitions[ch] = "q4"
        q3_output[ch] = "e"

    elif ch == "s":
        q3_transitions[ch] = "q7"
        q3_output[ch] = "s"

    elif ch == "":
        q3_transitions[ch] = "qend"
        q3_output[ch] = "+N+SG"

    else:
        q3_transitions[ch] = "q1"
        q3_output[ch] = ch

In [66]:
q6_transitions = {}
q6_output = {}


for ch in inputs:

    if ch == "":
        q6_transitions[ch] = "qend"
        q6_output[ch] = "+N+PL"

    elif ch == "e":
        q6_transitions[ch] = "q4"
        q6_output[ch] = "e"

    elif ch == "h":
        q6_transitions[ch] = "q3"
        q6_output[ch] = "h"

    else:
        q6_transitions[ch] = "q1"
        q6_output[ch] = ch

In [67]:
q4_transitions = {}
q4_output = {}


for ch in inputs:

    if ch == "s":
        q4_transitions[ch] = "q5"
        q4_output[ch] = "s"

    elif ch == "":
        q4_transitions[ch] = "qend"
        q4_output[ch] = "+N+SG"

    else:
        q4_transitions[ch] = "q1"
        q4_output[ch] = ch

In [68]:
q5_transitions = {}
q5_output = {}


for ch in inputs:

    if ch == "":
        q5_transitions[ch] = "qend"
        q5_output[ch] = "+N+PL"

    else:
        q5_transitions[ch] = "q1"
        q5_output[ch] = ch

In [69]:
q7_transitions = {}
q7_output = {}


for ch in inputs:

    if ch == "":
        q7_transitions[ch] = "qend"
        q7_output[ch] = "+invalid"

    else:
        q7_transitions[ch] = "q1"
        q7_output[ch] = ch

In [70]:
transitions = {

    "q1": q1_transitions,

    "q2": q2_transitions,

    "q3": q3_transitions,

    "q4": q4_transitions,

    "q5": q5_transitions,

    "q6": q6_transitions,

    "q7": q7_transitions
}

In [71]:
output_function = {

    "q1": q1_output,

    "q2": q2_output,

    "q3": q3_output,

    "q4": q4_output,

    "q5": q5_output,

    "q6": q6_output,

    "q7": q7_output
}

In [72]:
my_fst = FST(
    initial_state=initial_state,
    input_symbols=inputs,
    output_symbols=outputs,
    transition=transitions,
    output_function=output_function,
    final_state=final_state
)

In [73]:
test_words = [
    "fox",
    "foxes",
    "foxs",
    "bag",
    "bags",
    "watch",
    "watches",
    "watchs",
    "try",
    "tries",
]


for word in test_words:
    print(f"{word:<12} = {my_fst.check(word)}")

fox          = fox+N+SG
foxes        = fox+N+PL
foxs         = Invalid Word
bag          = bag+N+SG
bags         = bag+N+PL
watch        = watch+N+SG
watches      = watch+N+PL
watchs       = Invalid Word
try          = try+N+SG
tries        = try+N+PL


In [74]:
print(my_fst.check("irregularities"))

irregularity+N+PL


In [75]:
for word in data[:50]:
    print(f"{word} = {my_fst.check(word)}")

investigation = investigation+N+SG
primary = primary+N+SG
election = election+N+SG
evidence = evidence+N+SG
irregularities = irregularity+N+PL
place = place+N+SG
jury = jury+N+SG
presentments = Invalid Word
charge = charge+N+SG
election = election+N+SG
praise = praise+N+SG
thanks = Invalid Word
manner = manner+N+SG
election = election+N+SG
term = term+N+SG
jury = jury+N+SG
reports = report+N+PL
irregularities = irregularity+N+PL
primary = primary+N+SG
handful = handful+N+SG
reports = report+N+PL
jury = jury+N+SG
interest = interest+N+SG
election = election+N+SG
number = number+N+SG
voters = voter+N+PL
size = size+N+SG
city = city+N+SG
jury = jury+N+SG
registration = registration+N+SG
election = election+N+SG
laws = law+N+PL
legislators = legislator+N+PL
laws = law+N+PL
end = end+N+SG
jury = jury+N+SG
number = number+N+SG
topics = topics+N+SG
departments = department+N+PL
practices = practice+N+PL
interest = interest+N+SG
governments = government+N+PL
jury = jury+N+SG
offices = office+N

In [76]:
results = []

for word in data:

    result = my_fst.check(word)

    results.append(
        (word, result)
    )


print("Total processed words:", len(results))

Total processed words: 201654


In [77]:
output_file = Path("fst_output.txt")


with output_file.open("w", encoding="utf-8") as file:

    for word, result in results:

        file.write(
            f"{word} = {result}\n"
        )


print("Results saved to:", output_file)

Results saved to: fst_output.txt


In [78]:
print(my_fst.check("buses"))
print(my_fst.check("foxs"))

bus+N+PL
Invalid Word
